# faster-whisper / WhisperX

A refresher on the two libraries that make Whisper *production-grade*. **faster-whisper** is a drop-in reimplementation of OpenAI Whisper on the [CTranslate2](https://github.com/OpenNMT/CTranslate2) inference engine — same weights, same accuracy, but **4–5× faster** and a fraction of the memory thanks to quantization and a fused C++ kernel. **WhisperX** is a pipeline *on top* of faster-whisper that adds **word-level timestamps** (via forced alignment to a phoneme model) and **speaker diarization** (via `pyannote`), so you get "who said which word, exactly when."

**Domain:** Speech & Audio  ·  **recommended addition**  ·  **runnable:** yes  ·  _VAD + alignment demos run on CPU with no download; the real model call is gated behind `RUN_FASTER_WHISPER`_

## 1. What & Why

**What they are.** Plain `openai-whisper` is the correctness *reference*, but it's slow (pure PyTorch, fp32 on CPU) and gives you only coarse, segment-level timestamps. Two libraries fix that:

- **faster-whisper** (by SYSTRAN) runs the *exact same* Whisper checkpoints through **CTranslate2**, a Transformer inference engine that does operator fusion, batching, and **int8/fp16 quantization**. Result: ~4–5× the throughput, ~half the memory, and a built-in **VAD filter** (Silero) that strips silence before transcription. The API mirrors Whisper closely (`WhisperModel(...).transcribe(...)`).
- **WhisperX** (by Max Bain) wraps faster-whisper into a full pipeline: VAD-based chunking → batched transcription → **forced alignment** of every word to a wav2vec2 phoneme model (tight word timestamps) → optional **`pyannote` diarization** that tags each word with a speaker label.

**The problem they solve.** Vanilla Whisper's timestamps drift by hundreds of milliseconds (they're a *byproduct* of token generation, not a real alignment), it has no notion of speakers, and on a CPU or a long file it's painfully slow. faster-whisper solves *speed/memory*; WhisperX solves *precise word timing + speaker attribution*.

**When to reach for them.** faster-whisper: any time you transcribe at volume, on CPU, or with limited VRAM — it's the default production backend. WhisperX: subtitle/caption generation, karaoke-style word highlighting, meeting transcripts that need "who said what," or anything where a word being 300 ms off is visible.

**When not to.** True low-latency *streaming* (both are still chunk/batch oriented — use a streaming model). Ultra-simple one-off transcription where reference Whisper is fine. And WhisperX's diarization pulls in `pyannote` + a gated Hugging Face model, so it's heavier to set up than you might want for a quick job.

## 2. Mental Model

**Same Whisper brain, two upgrades bolted on: a faster engine underneath (faster-whisper) and an alignment + diarization stage on top (WhisperX).**

```
                         ┌──────────────── WhisperX pipeline ─────────────────┐
  audio ──▶ VAD ──▶ 30s chunks ──▶ faster-whisper ──▶ align ──▶ diarize ──▶ words
           (Silero)   (drop          (CTranslate2:    (wav2vec2   (pyannote:  + speaker
                       silence)       int8/fp16,       forced      who spoke   labels
                                      fused kernels)   alignment)  when)
                                          │
                       openai-whisper does THIS box in slow fp32 PyTorch and
                       emits only coarse segment timestamps — no align, no diarize.
```

Three ideas make it click:

1. **faster-whisper = Whisper weights on a different runtime.** Accuracy is unchanged; only the *engine* differs. Picking `compute_type="int8"` quantizes weights to 8-bit integers — the single biggest speed/memory lever, with negligible WER cost on `large`.
2. **VAD first.** Run voice-activity detection, *then* only transcribe the speech regions. Less audio through the model = faster runs *and* fewer silence-hallucinations.
3. **WhisperX timestamps come from alignment, not from Whisper.** Whisper's own timestamps are a guess; WhisperX throws them away and re-derives each word's start/end by force-aligning the transcript to the audio with a phoneme recognizer — that's why its word timing is tight.

## 3. Key Concepts

- **CTranslate2** — the C++/CUDA inference engine faster-whisper runs on. It fuses Transformer ops, supports dynamic batching, and quantizes weights, which is where the 4–5× speedup comes from. The model is shipped as a converted CT2 directory, not a PyTorch `.pt`.
- **`compute_type`** — the quantization/precision knob: `int8`, `int8_float16`, `float16`, `float32`. `int8` is the CPU default (smallest, fastest); `float16` for GPU; `int8_float16` mixes both on GPU. Lower precision = faster + smaller, with minor accuracy loss that's usually invisible on `large-v3`.
- **VAD filter** — `transcribe(..., vad_filter=True)` runs **Silero VAD** to detect speech and skips the gaps. Cuts compute on sparse audio and suppresses Whisper's classic silence hallucinations ("Thank you.", "Subtitles by…").
- **Segment generator** — faster-whisper's `transcribe()` returns a *lazy generator* of `Segment` objects plus an `info` struct (language, probability, duration). Nothing runs until you iterate — a common gotcha.
- **Forced alignment (WhisperX)** — given the transcript text and the audio, a **wav2vec2** phoneme model is aligned to the waveform (CTC alignment) to pin each *word* to a start/end time. This is independent of Whisper and far tighter than Whisper's native timestamps.
- **Diarization (WhisperX)** — `pyannote.audio` clusters speaker embeddings to answer "who spoke when," then WhisperX assigns each aligned word a `speaker` label. Needs a Hugging Face token and accepting the `pyannote` model terms.
- **`batch_size` (WhisperX)** — WhisperX batches the VAD-cut chunks through faster-whisper in parallel, the main reason it can hit 70× realtime on a GPU. faster-whisper alone transcribes sequentially per call.
- **Beam size / temperature / `condition_on_previous_text`** — same decoding knobs as reference Whisper; faster-whisper exposes them with the same names.

## 4. Setup

```bash
# faster-whisper: Whisper on CTranslate2 (CPU-friendly, int8 by default)
pip install -U faster-whisper

# WhisperX: word alignment + diarization on top of faster-whisper
pip install -U whisperx        # pulls in faster-whisper, torch, pyannote.audio
```

Both decode audio with the **ffmpeg** binary, so install it from your OS package manager
(`brew install ffmpeg` / `apt-get install ffmpeg`). Models download from the Hugging Face
hub on first use and cache locally; faster-whisper auto-converts them to the CTranslate2
format. WhisperX **diarization** additionally needs a Hugging Face token and acceptance of
the gated `pyannote/speaker-diarization` model terms.

To keep this notebook self-contained, the runnable cells below reproduce the two ideas
faster-whisper/WhisperX add — **VAD-based speech segmentation** and **word-level
alignment** — with only `numpy`, on synthetic audio: **no download, CPU-only**. The real
`WhisperModel` call is shown but gated behind `RUN_FASTER_WHISPER` so the notebook always
executes.

In [1]:
import os
import numpy as np

SR = 16000  # Whisper-family models always operate at 16 kHz mono
rng = np.random.default_rng(0)
print(f"numpy {np.__version__}  ·  sample rate {SR} Hz")
print("faster-whisper API shape:  from faster_whisper import WhisperModel")
print("whisperx API shape:        import whisperx")

numpy 2.5.0  ·  sample rate 16000 Hz
faster-whisper API shape:  from faster_whisper import WhisperModel
whisperx API shape:        import whisperx


## 5. Worked Examples

### Example 1 — Voice-activity detection: skip the silence (faster-whisper's `vad_filter`)

faster-whisper's headline speed/quality win on real recordings is the **VAD filter**: detect
the speech regions and only run those through the model. We synthesize 6 s of audio with two
speech bursts separated by silence, then run a tiny **energy-based VAD** — the same idea as
Silero, minus the neural net — to recover the spoken segments. Whisper then transcribes only
those windows, which is faster *and* avoids hallucinating text over the quiet gaps.

In [2]:
# Synthesize 6 s: speech-like bursts at 0.5-1.6 s and 3.4-4.7 s, silence elsewhere.
dur = 6.0
audio = 0.01 * rng.standard_normal(int(SR * dur)).astype(np.float32)  # quiet noise floor

def add_burst(sig, t0, t1, f0=180.0):
    """Drop a voiced-sounding burst (tone + harmonics) into [t0, t1) seconds."""
    i0, i1 = int(t0 * SR), int(t1 * SR)
    t = np.arange(i1 - i0) / SR
    burst = sum(0.5 / k * np.sin(2 * np.pi * f0 * k * t) for k in (1, 2, 3))
    sig[i0:i1] += (burst * np.hanning(len(t))).astype(np.float32)  # taper edges

add_burst(audio, 0.5, 1.6)
add_burst(audio, 3.4, 4.7, f0=220.0)

# Energy-based VAD: frame the signal, threshold short-time RMS, merge contiguous frames.
FRAME, HOP = int(0.025 * SR), int(0.010 * SR)   # 25 ms window, 10 ms hop
frames = np.lib.stride_tricks.sliding_window_view(audio, FRAME)[::HOP]
rms = np.sqrt((frames.astype(np.float64) ** 2).mean(axis=1))
thresh = rms.min() + 0.15 * (rms.max() - rms.min())   # adaptive gate
voiced = rms > thresh

# Collapse the boolean frame mask into (start, end) speech segments in seconds.
segments, start = [], None
for i, v in enumerate(voiced):
    if v and start is None:
        start = i
    elif not v and start is not None:
        segments.append((start * HOP / SR, i * HOP / SR))
        start = None
if start is not None:
    segments.append((start * HOP / SR, len(voiced) * HOP / SR))

speech = sum(e - s for s, e in segments)
print(f"total audio : {dur:.1f}s   ·   detected speech : {speech:.1f}s "
      f"({speech / dur:.0%}) -> Whisper skips the other {1 - speech / dur:.0%}")
for s, e in segments:
    print(f"  speech segment  {s:5.2f}s -> {e:5.2f}s  ({e - s:.2f}s)")

total audio : 6.0s   ·   detected speech : 1.7s (29%) -> Whisper skips the other 71%
  speech segment   0.64s ->  1.44s  (0.80s)
  speech segment   3.57s ->  4.51s  (0.94s)


### Example 2 — From segment timestamps to word timestamps (what WhisperX alignment buys)

Whisper gives you one coarse `(start, end)` per *segment*; WhisperX force-aligns the audio to
produce a tight `(start, end)` for every *word*. The real thing uses a wav2vec2 CTC model, but
the data transformation is the point: a segment splits into words whose durations are
proportional to their phonetic length, no longer guessed from token positions. Below we take a
Whisper-style segment and produce the per-word table WhisperX would emit — the structure your
subtitle/karaoke code actually consumes.

In [3]:
# A coarse, Whisper-style segment: just text + one (start, end) span.
segment = {"start": 3.40, "end": 4.70, "text": "the quick brown fox jumps"}

# WhisperX re-derives each word's timing from the audio. We approximate the *shape* of that
# output by weighting each word by its length (a stand-in for phoneme count / duration).
words = segment["text"].split()
weights = np.array([len(w) for w in words], dtype=float)
weights /= weights.sum()

span = segment["end"] - segment["start"]
edges = segment["start"] + np.concatenate([[0], np.cumsum(weights)]) * span

aligned = [
    {"word": w, "start": round(float(edges[i]), 2), "end": round(float(edges[i + 1]), 2),
     "score": round(0.80 + 0.18 * (1 - abs(0.5 - weights[i])), 2)}  # mock alignment confidence
    for i, w in enumerate(words)
]

print(f"input  segment : [{segment['start']:.2f}s -> {segment['end']:.2f}s]  \"{segment['text']}\"\n")
print("WhisperX word-level output:")
print(f"  {'word':<8}{'start':>7}{'end':>7}{'score':>7}")
for a in aligned:
    print(f"  {a['word']:<8}{a['start']:>7.2f}{a['end']:>7.2f}{a['score']:>7.2f}")

# This is exactly the structure WhisperX returns: result['segments'][i]['words'].
# Attach a speaker label per word and you have a diarized, word-timed transcript.
for a in aligned:
    a["speaker"] = "SPEAKER_00"
print(f"\nfirst word as WhisperX dict: {aligned[0]}")

input  segment : [3.40s -> 4.70s]  "the quick brown fox jumps"

WhisperX word-level output:
  word      start    end  score
  the        3.40   3.59   0.92
  quick      3.59   3.90   0.93
  brown      3.90   4.20   0.93
  fox        4.20   4.39   0.92
  jumps      4.39   4.70   0.93

first word as WhisperX dict: {'word': 'the', 'start': 3.4, 'end': 3.59, 'score': np.float64(0.92), 'speaker': 'SPEAKER_00'}


### Example 3 — The real transcription call (gated behind `RUN_FASTER_WHISPER`)

With the library installed, faster-whisper is a few lines. The model download is ~75 MB for
`tiny` (and converts to CTranslate2 on first load), so this is gated — set
`RUN_FASTER_WHISPER=1` to run it for real on the synthetic audio from Example 1. Either way the
cell prints the canonical call shapes for **faster-whisper** and the full **WhisperX** pipeline.

In [4]:
if os.getenv("RUN_FASTER_WHISPER"):
    from faster_whisper import WhisperModel
    # compute_type="int8" is the fast, low-memory CPU default; use "float16" on GPU.
    model = WhisperModel("tiny", device="cpu", compute_type="int8")
    # transcribe() returns a LAZY generator + an info struct — nothing runs until you iterate.
    segments, info = model.transcribe(audio, vad_filter=True, beam_size=5)
    print(f"language={info.language} (p={info.language_probability:.2f})")
    for seg in segments:  # iterating is what actually drives inference
        print(f"  [{seg.start:5.2f}s -> {seg.end:5.2f}s] {seg.text}")
else:
    print("Set RUN_FASTER_WHISPER=1 to download & run Whisper-tiny via CTranslate2 (~75 MB).\n")
    print("faster-whisper:")
    print('    from faster_whisper import WhisperModel')
    print('    model = WhisperModel("large-v3", device="cuda", compute_type="float16")')
    print('    segments, info = model.transcribe("audio.mp3", vad_filter=True, beam_size=5)')
    print('    for s in segments:               # lazy generator -> iterate to transcribe')
    print('        print(s.start, s.end, s.text)\n')
    print("WhisperX (transcribe -> align -> diarize):")
    print('    import whisperx')
    print('    audio = whisperx.load_audio("audio.mp3")')
    print('    model = whisperx.load_model("large-v3", device="cuda", compute_type="float16")')
    print('    result = model.transcribe(audio, batch_size=16)')
    print('    a_model, meta = whisperx.load_align_model(result["language"], device="cuda")')
    print('    result = whisperx.align(result["segments"], a_model, meta, audio, "cuda")')
    print('    # result["segments"][i]["words"] -> [{word, start, end, score}, ...]')
    print('    dia = whisperx.diarize.DiarizationPipeline(use_auth_token=HF_TOKEN, device="cuda")')
    print('    result = whisperx.assign_word_speakers(dia(audio), result)  # adds "speaker"')

Set RUN_FASTER_WHISPER=1 to download & run Whisper-tiny via CTranslate2 (~75 MB).

faster-whisper:
    from faster_whisper import WhisperModel
    model = WhisperModel("large-v3", device="cuda", compute_type="float16")
    segments, info = model.transcribe("audio.mp3", vad_filter=True, beam_size=5)
    for s in segments:               # lazy generator -> iterate to transcribe
        print(s.start, s.end, s.text)

WhisperX (transcribe -> align -> diarize):
    import whisperx
    audio = whisperx.load_audio("audio.mp3")
    model = whisperx.load_model("large-v3", device="cuda", compute_type="float16")
    result = model.transcribe(audio, batch_size=16)
    a_model, meta = whisperx.load_align_model(result["language"], device="cuda")
    result = whisperx.align(result["segments"], a_model, meta, audio, "cuda")
    # result["segments"][i]["words"] -> [{word, start, end, score}, ...]
    dia = whisperx.diarize.DiarizationPipeline(use_auth_token=HF_TOKEN, device="cuda")
    result = whisper

## 6. Gotchas & Pitfalls

- **`transcribe()` returns a lazy generator.** faster-whisper does *nothing* until you iterate
  the `segments`. `segments, info = model.transcribe(...)` is instant; the work happens in the
  `for seg in segments:` loop. Wrap in `list(segments)` if you need it all up front.
- **`compute_type` mismatch falls back silently.** Ask for `float16` on a CPU and CTranslate2
  quietly downgrades to `float32` (slower). On CPU use `int8`; on GPU use `float16` or
  `int8_float16`. Check the logged effective compute type.
- **VAD can clip speech.** `vad_filter=True` is great for hallucinations, but aggressive VAD
  settings drop quiet word-onsets. Tune `vad_parameters` (e.g. `min_silence_duration_ms`) if
  words go missing at segment edges.
- **WhisperX alignment fails on languages without an align model.** Forced alignment needs a
  wav2vec2 phoneme model for that language; unsupported languages silently fall back to coarse
  timestamps. Numbers/symbols ("$5", "2024") also align poorly — normalize text first.
- **Diarization is gated and heavy.** WhisperX diarization needs a Hugging Face token *and*
  acceptance of the `pyannote/speaker-diarization-3.1` terms, pulls in `pyannote.audio`, and
  wants the number of speakers (`min_speakers`/`max_speakers`) to behave well.
- **Don't expect a free lunch on accuracy.** faster-whisper is the *same weights* as Whisper —
  it is not more accurate, just faster. `int8` can nudge WER up slightly on small models; it's
  negligible on `large-v3`.
- **Still not streaming.** Both are chunk/batch oriented. For live captions you need a true
  streaming setup (overlapping windows, partial hypotheses) — these libraries don't do it.
- **Version coupling.** WhisperX pins specific faster-whisper / CTranslate2 / torch versions;
  mismatches surface as cryptic CUDA or cuDNN load errors. Install WhisperX into a clean env.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off |
|---|---|---|
| **openai-whisper (reference)** | Correctness baseline, research, the canonical CLI | Slowest; fp32 PyTorch; coarse timestamps; no diarization |
| **faster-whisper** | Production batch transcription, CPU/low-VRAM, throughput | 4–5× faster, built-in VAD — but a converted CT2 model and a lazy generator API |
| **WhisperX** | Precise word timestamps + "who said what" (subtitles, meetings) | Adds wav2vec2 alignment + `pyannote`; heavier, gated diarization model, version-sensitive |
| **whisper.cpp** | On-device / edge, no Python, quantized GGUF | C++ build; fewer knobs; no built-in diarization |
| **NVIDIA NeMo / Parakeet** | Streaming, ultra-low latency, fine-tuning | More setup; less zero-shot multilingual robustness than Whisper |
| **Cloud ASR (Deepgram, AssemblyAI, AWS)** | Managed scale, real-time SDKs, built-in diarization, SLAs | Per-minute cost; audio leaves your machine; vendor lock-in |

**Rule of thumb:** start from reference Whisper to validate accuracy, then switch to
**faster-whisper** the moment speed or memory matters (almost always). Add **WhisperX** when
you need word-accurate timing or speaker labels. Reach past both only for true streaming
(NeMo/streaming model) or when a managed cloud SLA beats running it yourself.

## 8. Resources

- **faster-whisper (GitHub)** — install, `WhisperModel` API, VAD, benchmarks: https://github.com/SYSTRAN/faster-whisper
- **WhisperX (GitHub)** — transcribe → align → diarize pipeline, examples: https://github.com/m-bain/whisperX
- **WhisperX paper — *Time-Accurate Speech Transcription of Long-Form Audio*** (Bain et al., 2023): https://arxiv.org/abs/2303.00747
- **CTranslate2 docs** — the inference engine, quantization, `compute_type`: https://opennmt.net/CTranslate2/
- **Whisper (reference)** — the underlying model and weights: https://github.com/openai/whisper
- **Companion notebook** — `whisper-stt.ipynb` for the core Whisper model and its log-mel front end.